# 1. Imports, settings, and data download

This section downloads the first version of the intraday dataset.

For the preliminary EDA, we use:

- `^GSPC` as the SPX index price series
- `^VIX` as a proxy for implied volatility conditions
- `SPY` as a proxy for volume and VWAP, because SPX itself is an index and does not have meaningful trading volume

Yahoo Finance is used for now because it is simple and free. The limitation is that 15-minute intraday data is usually only available for a short recent window, so this is suitable for initial EDA rather than the final dissertation dataset.

In [ ]:
# Basic libraries
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Modelling libraries used later in the notebook
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Optional stationarity test used later
from statsmodels.tsa.stattools import adfuller

# Save files into a local data folder
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Tickers
TICKER_SPX = "^GSPC"
TICKER_VIX = "^VIX"
TICKER_SPY = "SPY"

# Yahoo Finance settings
PERIOD = "60d"
INTERVAL = "15m"

# Market session settings
TZ = "America/New_York"
SESSION_START = "09:30"
SESSION_END = "16:00"

# We enter at 3:00pm using the latest fully available 15-minute bar.
# In 15-minute Yahoo data, the 14:45 bar represents 14:45 to 15:00.
ENTRY_BAR_TIME = "14:45:00"

# The 15:45 bar represents 15:45 to 16:00, so its close is used as the final close.
CLOSE_BAR_TIME = "15:45:00"

RANDOM_STATE = 42

## Download intraday data

We now download SPX, VIX, and SPY separately.

At this early EDA stage, the code is intentionally kept simple. More robust error handling can be added later when the data pipeline becomes more permanent.

In [ ]:
spx_raw = yf.download(
    TICKER_SPX,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=True,
    progress=False
)

vix_raw = yf.download(
    TICKER_VIX,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=True,
    progress=False
)

spy_raw = yf.download(
    TICKER_SPY,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=True,
    progress=False
)

: 

## Basic timestamp and session cleaning

Yahoo returns timestamps in an exchange-aware format. We convert everything to New York time and keep only the regular U.S. trading session from 9:30am to 4:00pm.

In [ ]:
# Make column names easier to work with
spx_raw.columns = spx_raw.columns.str.lower()
vix_raw.columns = vix_raw.columns.str.lower()
spy_raw.columns = spy_raw.columns.str.lower()

# Convert timestamps to New York time
spx_raw.index = spx_raw.index.tz_convert(TZ)
vix_raw.index = vix_raw.index.tz_convert(TZ)
spy_raw.index = spy_raw.index.tz_convert(TZ)

# Keep regular market hours only
spx_raw = spx_raw.between_time(SESSION_START, SESSION_END, inclusive="left")
vix_raw = vix_raw.between_time(SESSION_START, SESSION_END, inclusive="left")
spy_raw = spy_raw.between_time(SESSION_START, SESSION_END, inclusive="left")

# Label the index clearly
spx_raw.index.name = "datetime"
vix_raw.index.name = "datetime"
spy_raw.index.name = "datetime"

print("SPX:", spx_raw.shape, spx_raw.index.min(), "to", spx_raw.index.max())
print("VIX:", vix_raw.shape, vix_raw.index.min(), "to", vix_raw.index.max())
print("SPY:", spy_raw.shape, spy_raw.index.min(), "to", spy_raw.index.max())

spx_raw.head()